Cell 1: setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

XLSX_PATH = "combined_ed.xlsx"
CSV_PATH = "combined_ed.csv"

Cell 2: check sheets and convert to CSV

In [2]:
# Read every sheet so nothing gets silently dropped
sheets = pd.read_excel(XLSX_PATH, sheet_name=None)

print("Sheets found:", list(sheets.keys()))
for name, sheet_df in sheets.items():
    print(f"  {name}: {sheet_df.shape[0]} rows x {sheet_df.shape[1]} cols")

# Stack sheets (if there is more than one), keep track of where each row came from
raw = pd.concat(
    [s.assign(source_sheet=name) for name, s in sheets.items()],
    ignore_index=True
)

raw.to_csv(CSV_PATH, index=False)
print(f"\nSaved {CSV_PATH} with {len(raw)} rows")

Sheets found: ['Sheet1']
  Sheet1: 11760 rows x 12 cols

Saved combined_ed.csv with 11760 rows


Cell 3: reload from CSV and basic structure

In [3]:
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())
df.head(10)

Shape: (11760, 13)

Columns: ['#', 'WFO [deg]', 'WWR [-]', 'WDH [m]', 'SDT [-]', 'SDN [-]', 'SDD [m]', 'SDA [deg]', 'Cooling Demand [kWh/m2]', 'Heating Demand [kWh/m2]', 'Electricity Use [kWh/m2]', 'Total Energy Demand [kWh/m2]', 'source_sheet']

Dtypes:
#                                 int64
WFO [deg]                         int64
WWR [-]                         float64
WDH [m]                         float64
SDT [-]                             str
SDN [-]                           int64
SDD [m]                         float64
SDA [deg]                         int64
Cooling Demand [kWh/m2]         float64
Heating Demand [kWh/m2]         float64
Electricity Use [kWh/m2]        float64
Total Energy Demand [kWh/m2]    float64
source_sheet                        str
dtype: object

Missing values per column:
#                               0
WFO [deg]                       0
WWR [-]                         0
WDH [m]                         0
SDT [-]                         0
SDN [-]      

,#,WFO [deg],WWR [-],WDH [m],SDT [-],SDN [-],SDD [m],SDA [deg],Cooling Demand [kWh/m2],Heating Demand [kWh/m2],Electricity Use [kWh/m2],Total Energy Demand [kWh/m2],source_sheet
0,0,0,0.15,1.66,HSD,1,0.3,90,1.081554e+11,1.205674e+11,1.648504e+09,2.303713e+11,Sheet1
1,1,45,0.15,1.66,HSD,1,0.3,90,1.080874e+11,1.204722e+11,1.514692e+09,2.300743e+11,Sheet1
2,2,90,0.15,1.66,HSD,1,0.3,90,1.080167e+11,1.203463e+11,1.454610e+09,2.298175e+11,Sheet1
3,3,135,0.15,1.66,HSD,1,0.3,90,1.081876e+11,1.203604e+11,1.450462e+09,2.299984e+11,Sheet1
4,4,180,0.15,1.66,HSD,1,0.3,90,1.083706e+11,1.204864e+11,1.545570e+09,2.304026e+11,Sheet1
5,5,225,0.15,1.66,HSD,1,0.3,90,1.082026e+11,1.205496e+11,1.730796e+09,2.304830e+11,Sheet1
6,6,270,0.15,1.66,HSD,1,0.3,90,1.080961e+11,1.205330e+11,1.924333e+09,2.305535e+11,Sheet1
7,7,315,0.15,1.66,HSD,1,0.3,90,1.081270e+11,1.205499e+11,1.859322e+09,2.305363e+11,Sheet1
8,8,0,0.20,1.48,HSD,1,0.3,90,1.083140e+11,1.205302e+11,1.332201e+09,2.301765e+11,Sheet1
9,9,45,0.20,1.48,HSD,1,0.3,90,1.082500e+11,1.203260e+11,1.263027e+09,2.298391e+11,Sheet1


Cell 4: unique values of the inputs

In [4]:
input_cols = [c for c in df.columns if any(k in c for k in ["WFO", "WWR", "WDH", "SDT", "SDN", "SDD", "SDA"])]

for c in input_cols:
    vals = sorted(df[c].dropna().unique().tolist(), key=str)
    print(f"{c}  ({len(vals)} unique)")
    print("   ", vals)
    print()

WFO [deg]  (8 unique)
    [0, 135, 180, 225, 270, 315, 45, 90]

WWR [-]  (7 unique)
    [0.15000000000000005, 0.20000000000000004, 0.25, 0.3, 0.35000000000000003, 0.4, 0.45]

WDH [m]  (7 unique)
    [0.58, 0.76, 0.94, 1.12, 1.3, 1.48, 1.66]

SDT [-]  (2 unique)
    ['HSD', 'VSD']

SDN [-]  (4 unique)
    [1, 2, 4, 6]

SDD [m]  (5 unique)
    [0.3, 0.6, 0.9, 1.2, 1.5]

SDA [deg]  (9 unique)
    [105, 120, 135, 150, 30, 45, 60, 75, 90]



Cell 5: duplicates and conflicting repeats

In [5]:
print("Fully duplicated rows:", df.duplicated().sum())

# Repeated configurations (same inputs) - do they have different outputs?
config_dups = df.duplicated(subset=input_cols, keep=False).sum()
print("Rows sharing an identical input configuration:", config_dups)

if config_dups > 0:
    cool_col = [c for c in df.columns if "Cooling" in c][0]
    conflicts = (df[df.duplicated(subset=input_cols, keep=False)]
                 .groupby(input_cols)[cool_col].nunique())
    print("Configurations with conflicting cooling values:", (conflicts > 1).sum())

Fully duplicated rows: 0
Rows sharing an identical input configuration: 0


Cell 6: is the full grid present, and how is “no shading” represented?

In [6]:
# Expected rows if every combination of inputs exists exactly once
# (WDH is excluded because it is derived from WWR)
grid_cols = [c for c in input_cols if "WDH" not in c]
expected = np.prod([df[c].nunique() for c in grid_cols])
print("Rows in file:", len(df))
print("Product of unique values (full grid):", expected)

print("\nRows per shading type:")
print(df[[c for c in df.columns if "SDT" in c][0]].value_counts())

# Look for a 'no shading' encoding
sdt, sdn, sdd, sda = [[c for c in df.columns if k in c][0] for k in ["SDT", "SDN", "SDD", "SDA"]]
print("\nSDT x SDN:")
print(pd.crosstab(df[sdt], df[sdn]))
print("\nSDT x SDA:")
print(pd.crosstab(df[sdt], df[sda]))
print("\nSDT x SDD:")
print(pd.crosstab(df[sdt], df[sdd]))

Rows in file: 11760
Product of unique values (full grid): 20160

Rows per shading type:
SDT [-]
VSD    7560
HSD    4200
Name: count, dtype: int64

SDT x SDN:
SDN [-]     1     2     4     6
SDT [-]                        
HSD      1400  1400  1400     0
VSD         0  2520  2520  2520

SDT x SDA:
SDA [deg]  30   45   60   75   90   105  120  135  150
SDT [-]                                               
HSD          0    0    0    0  840  840  840  840  840
VSD        840  840  840  840  840  840  840  840  840

SDT x SDD:
SDD [m]   0.3   0.6   0.9   1.2   1.5
SDT [-]                              
HSD       840   840   840   840   840
VSD      1512  1512  1512  1512  1512


Cell 7: output sanity check

In [7]:
out_cols = [c for c in df.columns if any(k in c for k in ["Cooling", "Heating", "Electricity", "Total"])]
print(df[out_cols].describe().T)

cool = df[[c for c in out_cols if "Cooling" in c][0]]
heat = df[[c for c in out_cols if "Heating" in c][0]]
elec = df[[c for c in out_cols if "Electricity" in c][0]]
total = df[[c for c in out_cols if "Total" in c][0]]

print("\nTotal == Cooling + Heating ?          ", np.allclose(total, cool + heat, rtol=0.01))
print("Total == Cooling + Heating + Elec ?   ", np.allclose(total, cool + heat + elec, rtol=0.01))

print("\nCooling value range (min, max):", cool.min(), cool.max())
print("Relative spread (max-min)/mean: ", round((cool.max() - cool.min()) / cool.mean() * 100, 2), "%")

                                count          mean           std           min           25%           50%           75%           max
Cooling Demand [kWh/m2]       11760.0  1.088051e+11  5.246865e+08  1.077505e+11  1.084365e+11  1.086776e+11  1.090423e+11  1.114405e+11
Heating Demand [kWh/m2]       11760.0  1.203088e+11  3.079722e+08  1.186432e+11  1.202441e+11  1.204292e+11  1.204838e+11  1.207975e+11
Electricity Use [kWh/m2]      11760.0  1.814711e+09  6.289362e+08  9.501081e+08  1.239627e+09  1.689469e+09  2.355754e+09  3.048828e+09
Total Energy Demand [kWh/m2]  11760.0  2.309286e+11  6.678743e+08  2.291603e+11  2.304269e+11  2.308767e+11  2.314807e+11  2.327293e+11

Total == Cooling + Heating ?           False
Total == Cooling + Heating + Elec ?    True

Cooling value range (min, max): 107750526745.735 111440516795.956
Relative spread (max-min)/mean:  3.39 %


Cell 8: clean and rename

In [8]:
data = df.rename(columns={
    "WFO [deg]": "wfo", "WWR [-]": "wwr", "WDH [m]": "wdh",
    "SDT [-]": "sdt", "SDN [-]": "sdn", "SDD [m]": "sdd", "SDA [deg]": "sda",
    "Cooling Demand [kWh/m2]": "cooling",
})[["wfo", "wwr", "wdh", "sdt", "sdn", "sdd", "sda", "cooling"]].copy()

data["wwr"] = data["wwr"].round(2)   # removes 0.15000000000000005 style noise

# WWR and WDH should be a perfect one-to-one pair
print("WWR -> WDH mapping:")
print(data.groupby("wwr")["wdh"].nunique().to_dict())
print(data[["wwr", "wdh"]].drop_duplicates().sort_values("wwr").to_string(index=False))

data = data.drop(columns="wdh")
data.to_csv("clean_cooling.csv", index=False)
data.head()

WWR -> WDH mapping:
{0.15: 1, 0.2: 1, 0.25: 1, 0.3: 1, 0.35: 1, 0.4: 1, 0.45: 1}
 wwr  wdh
0.15 1.66
0.20 1.48
0.25 1.30
0.30 1.12
0.35 0.94
0.40 0.76
0.45 0.58


,wfo,wwr,sdt,sdn,sdd,sda,cooling
0,0,0.15,HSD,1,0.3,90,1.081554e+11
1,45,0.15,HSD,1,0.3,90,1.080874e+11
2,90,0.15,HSD,1,0.3,90,1.080167e+11
3,135,0.15,HSD,1,0.3,90,1.081876e+11
4,180,0.15,HSD,1,0.3,90,1.083706e+11


Cell 9: which inputs move cooling?

In [9]:
overall = data["cooling"].mean()
for c in ["wfo", "wwr", "sdt", "sdn", "sdd", "sda"]:
    means = data.groupby(c)["cooling"].mean()
    spread = (means.max() - means.min()) / overall * 100
    print(f"{c}: spread of group means = {spread:.2f}% of overall mean")
    print((means / overall * 100 - 100).round(2).to_string())
    print()

wfo: spread of group means = 0.62% of overall mean
wfo
0      0.00
45    -0.02
90    -0.09
135    0.19
180    0.37
225   -0.02
270   -0.25
315   -0.19

wwr: spread of group means = 0.89% of overall mean
wwr
0.15   -0.42
0.20   -0.30
0.25   -0.17
0.30   -0.02
0.35    0.14
0.40    0.30
0.45    0.47

sdt: spread of group means = 0.39% of overall mean
sdt
HSD   -0.25
VSD    0.14

sdn: spread of group means = 0.25% of overall mean
sdn
1   -0.18
2    0.07
4   -0.03
6    0.05

sdd: spread of group means = 0.10% of overall mean
sdd
0.3    0.07
0.6   -0.01
0.9   -0.02
1.2   -0.02
1.5   -0.02

sda: spread of group means = 0.28% of overall mean
sda
30     0.11
45     0.13
60     0.14
75     0.17
90    -0.00
105   -0.05
120   -0.06
135   -0.05
150   -0.11



Cell 10: unit check (joules assumption)

In [ ]:
kwh_m2 = data["cooling"] / 3.6e6 / 48
print(kwh_m2.describe())

count    11760.000000
mean       629.659111
std          3.036380
min        623.556289
25%        627.525770
50%        628.921504
75%        631.031568
max        644.910398
Name: cooling, dtype: float64


: 